In [1]:
!pip install yfinance pandas matplotlib --quiet

In [2]:
import yfinance as yf
import pandas as pd
# for BSM
from datetime import datetime
from scipy.stats import norm
import numpy as np

In [3]:
stock = yf.Ticker("AAPL")
stock.options

('2026-07-24',
 '2026-07-27',
 '2026-07-29',
 '2026-07-31',
 '2026-08-07',
 '2026-08-14',
 '2026-08-21',
 '2026-08-28',
 '2026-09-18',
 '2026-10-16',
 '2026-11-20',
 '2026-12-18',
 '2027-01-15',
 '2027-02-19',
 '2027-03-19',
 '2027-06-17',
 '2027-09-17',
 '2027-12-17',
 '2028-01-21',
 '2028-03-17',
 '2028-12-15')

In [4]:
stock.history()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-06-23 00:00:00-04:00,297.540009,301.640015,294.179993,294.299988,52010900,0.0,0.0
2026-06-24 00:00:00-04:00,295.359985,299.700012,292.940002,293.079987,53081900,0.0,0.0
2026-06-25 00:00:00-04:00,287.399994,288.799988,273.750000,275.149994,107013700,0.0,0.0
2026-06-26 00:00:00-04:00,275.000000,285.950012,274.209991,283.779999,261775500,0.0,0.0
2026-06-29 00:00:00-04:00,286.730011,288.369995,279.850006,281.739990,66427000,0.0,0.0
2026-06-30 00:00:00-04:00,281.170013,289.940002,280.700012,289.359985,65100200,0.0,0.0
2026-07-01 00:00:00-04:00,293.440002,296.589996,289.200012,294.380005,50164200,0.0,0.0
2026-07-02 00:00:00-04:00,294.119995,309.420013,293.679993,308.630005,75352800,0.0,0.0
2026-07-06 00:00:00-04:00,307.359985,314.200012,307.000000,312.660004,53590000,0.0,0.0


In [5]:
#PULLING OPTION CHAIN DATA OF SPECIFIC EXPIRY
#I am doing of july and september
chain = stock.option_chain("2026-07-31")
calls_df1 = chain.calls
puts_df1 = chain.puts

chain2 = stock.option_chain("2026-09-18")
calls_df2= chain2.calls
puts_df2 = chain2.puts

In [6]:
S = stock.history(period="1d")['Close'].iloc[-1] #spotprice
print(S)
calls_df1.info()

325.8900146484375
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   contractSymbol     73 non-null     object             
 1   lastTradeDate      73 non-null     datetime64[ns, UTC]
 2   strike             73 non-null     float64            
 3   lastPrice          73 non-null     float64            
 4   bid                73 non-null     float64            
 5   ask                73 non-null     float64            
 6   change             73 non-null     float64            
 7   percentChange      73 non-null     float64            
 8   volume             73 non-null     int64              
 9   openInterest       73 non-null     int64              
 10  impliedVolatility  73 non-null     float64            
 11  inTheMoney         73 non-null     bool               
 12  contractSize       73 non-null    

In [7]:
#function for calculating the difference between strike price and spot stock price
# Categorizing ITM, ATM, OTM
def classifictaion(df, S, op_type):
    df = df.copy()
    df['difference'] = (df['strike'] - S).abs()
    if op_type == 'call':
        itm_op = df['strike'] < S
        otm_op = df['strike'] > S
    elif op_type == 'put' :
        itm_op = df['strike'] > S
        otm_op = df['strike'] < S

    itm = df[itm_op].sort_values('difference').head(10)
    otm = df[otm_op].sort_values('difference').head(10)
    atm = df.loc[[df['difference'].idxmin()]] #the most appropriate at the money options data

    return itm, otm, atm

itm_c1, otm_c1, atm_c1 = classifictaion(calls_df1, S, 'call')
itm_c2, otm_c2, atm_c2 = classifictaion(calls_df2, S, 'call')
itm_p1, otm_p1, atm_p1 = classifictaion(puts_df1, S, 'put')
itm_p2, otm_p2, atm_p2 = classifictaion(calls_df1, S, 'put')

In [8]:
#Task 3 : BSM method
today = datetime.now()
expiry1 = datetime(2026,7,31)
expiry2 = datetime(2026,7,31)
T1 = (expiry1 - today).days / 365
T2 = (expiry2 - today).days / 365
def bsm(S, K, T, r,sigma, op_type):
    d1 = (np.log(S/K) + (r+ 0.5* sigma**2)*T) / (sigma * np.sqrt(T))
    d2 = d1 - (sigma * np.sqrt(T))
    if op_type == 'call':
        price = S * norm.cdf(d1) - K* np.exp(-r * T)*norm.cdf(d2)
    else:
        price =  K* np.exp(-r * T)*norm.cdf(-d2) - S * norm.cdf(-d1) 
    return price

In [16]:
# create a dictionary
options_info ={
    'ITM calls1': (itm_c1,T1, 'call'),
    'OTM calls1': (otm_c1,T1, 'call'),
    'ATM calls1': (atm_c1,T1, 'call'),
    'ITM calls2': (itm_c1,T2, 'call'),
    'OTM calls2': (otm_c1,T2, 'call'),
    'ATM calls2': (atm_c1,T2, 'call'),
    'ITM puts1': (itm_p1,T1, 'put'),
    'OTM puts1': (otm_p1,T1, 'put'),
    'ATM puts1': (atm_p1,T1, 'put'),
}

r = 0.038 # current 3-month yield on US Treasury bill
for name, (df, T, op_type) in options_info.items():
    df['BSM price'] = df.apply(
        lambda row : bsm(S, row['strike'], T, r, row['impliedVolatility'], op_type),
        axis = 1
    )
    print(name)
    print(df[['strike', 'lastPrice', 'BSM price']])

ITM calls1
    strike  lastPrice  BSM price
47   325.0       8.69   1.126778
46   322.5      10.10   3.624956
45   320.0      11.59   6.123135
44   317.5      13.15   8.621314
43   315.0      15.02  11.119493
42   312.5      14.97  13.617671
41   310.0      19.02  16.115850
40   307.5      19.34  18.614029
39   305.0      22.60  21.112208
38   302.5      23.22  23.610386
OTM calls1
    strike  lastPrice     BSM price
48   327.5       7.22  4.339995e-06
49   330.0       6.20  3.059216e-09
50   332.5       5.20  1.144133e-06
51   335.0       4.23  6.190185e-11
52   337.5       3.44  2.450761e-05
53   340.0       2.80  4.147017e-07
54   342.5       2.22  3.689634e-09
55   345.0       1.82  1.733014e-11
56   347.5       1.52  1.733417e-04
57   350.0       1.10  2.911426e-05
ATM calls1
    strike  lastPrice  BSM price
47   325.0       8.69   1.126778
ITM calls2
    strike  lastPrice  BSM price
47   325.0       8.69   1.126778
46   322.5      10.10   3.624956
45   320.0      11.59   6.123135

In [23]:
for name, (df, T, op_type) in options_info.items():
    df['Diff_btw_lstp_bsm'] = df['lastPrice'] - df['BSM price']
    df['status'] = df['Diff_btw_lstp_bsm'].apply(lambda x: 'Overpriced' if x>0 else 'Underpriced')
    print(name)
    print(df[[ 'lastPrice', 'BSM price', 'status']])

ITM calls1
    lastPrice  BSM price       status
47       8.69   1.126778   Overpriced
46      10.10   3.624956   Overpriced
45      11.59   6.123135   Overpriced
44      13.15   8.621314   Overpriced
43      15.02  11.119493   Overpriced
42      14.97  13.617671   Overpriced
41      19.02  16.115850   Overpriced
40      19.34  18.614029   Overpriced
39      22.60  21.112208   Overpriced
38      23.22  23.610386  Underpriced
OTM calls1
    lastPrice     BSM price      status
48       7.22  4.339995e-06  Overpriced
49       6.20  3.059216e-09  Overpriced
50       5.20  1.144133e-06  Overpriced
51       4.23  6.190185e-11  Overpriced
52       3.44  2.450761e-05  Overpriced
53       2.80  4.147017e-07  Overpriced
54       2.22  3.689634e-09  Overpriced
55       1.82  1.733014e-11  Overpriced
56       1.52  1.733417e-04  Overpriced
57       1.10  2.911426e-05  Overpriced
ATM calls1
    lastPrice  BSM price      status
47       8.69   1.126778  Overpriced
ITM calls2
    lastPrice  BSM price